# Wind power regression: portfolio-review revision

**This notebook is a later portfolio-review revision, not part of the original analysis.** It was written by Kudzai Shava (with AI assistance for the cleanup) to correct a problem found in the original regression section of `wind_scada.ipynb`.

**What was wrong in the original:** `LV ActivePower (kW)` was both the target and one of the input features, and the data was scaled and split randomly. The reported errors (about 1e-31 to 1e-5 on a scaled target) therefore reflect copying an input to the output, not predictive skill.

**What changes here:**
- The target is removed from the features. Inputs are wind speed, wind direction and the manufacturer's theoretical power curve value.
- The split is chronological: first 80% of rows train, last 20% test. No shuffling.
- The scaler is fitted on the training rows only. The target is left in kW so errors are interpretable.
- Random seed 42 is set for the Random Forest.
- A naive baseline (predict the theoretical power curve value directly) is added for context.

**What stays the same:** same dataset (`../data/T1.csv`), same two estimators (Linear Regression, Random Forest with 100 trees). The "IDK" uncertainty filter is not reproduced.

**Limits:** one turbine, one year, one split. No cross-validation, no tuning, no failure labels. Errors describe how well power can be estimated from wind conditions on later data; they say nothing about fault detection.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/T1.csv')
df['Date/Time'] = pd.to_datetime(df['Date/Time'], format='%d %m %Y %H:%M')
df = df.sort_values('Date/Time').reset_index(drop=True)
print(df.shape, df['Date/Time'].min(), df['Date/Time'].max())

(50530, 5) 2018-01-01 00:00:00 2018-12-31 23:50:00


In [2]:
features = ['Wind Speed (m/s)', 'Wind Direction (°)', 'Theoretical_Power_Curve (KWh)']
target = 'LV ActivePower (kW)'
assert target not in features

split = int(len(df) * 0.8)            # chronological split, no shuffling
train, test = df.iloc[:split], df.iloc[split:]
print('Train:', train['Date/Time'].min(), 'to', train['Date/Time'].max(), len(train))
print('Test: ', test['Date/Time'].min(), 'to', test['Date/Time'].max(), len(test))

scaler = StandardScaler().fit(train[features])    # fitted on training rows only
X_train, X_test = scaler.transform(train[features]), scaler.transform(test[features])
y_train, y_test = train[target].values, test[target].values

Train: 2018-01-01 00:00:00 to 2018-10-18 22:30:00 40424
Test:  2018-10-18 22:40:00 to 2018-12-31 23:50:00 10106


In [3]:
def score(name, pred):
    return {'Model': name,
            'MAE (kW)': mean_absolute_error(y_test, pred),
            'RMSE (kW)': np.sqrt(mean_squared_error(y_test, pred)),
            'R²': r2_score(y_test, pred)}

rows = [score('Baseline: theoretical power curve', test['Theoretical_Power_Curve (KWh)'].values)]
for name, model in [('Linear Regression', LinearRegression()),
                    ('Random Forest (100 trees, seed 42)', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))]:
    model.fit(X_train, y_train)
    rows.append(score(name, model.predict(X_test)))
results = pd.DataFrame(rows).round(3)
results

,Model,MAE (kW),RMSE (kW),R²
0,Baseline: theoretical power curve,207.903,470.712,0.877
1,Linear Regression,221.295,414.466,0.905
2,"Random Forest (100 trees, seed 42)",266.847,556.558,0.828


## Reading the results

Executed once in the environment recorded in `../docs/technical-review.md` (outputs above). On the chronological hold-out (last 20% of rows, 18 Oct to 31 Dec 2018):

- The original notebook's near-zero errors do not reappear once the target is removed from the inputs. Errors are now in the hundreds of kW (turbine output ranges up to about 3,600 kW).
- Linear Regression has the lowest RMSE and highest R². The raw manufacturer power curve has the lowest MAE. Random Forest is the weakest of the three on this split.
- This is one split on one turbine with default settings. It is not evidence about which model is better in general, and it says nothing about fault detection.